In [1]:
from pathlib import Path

import kagglehub
import numpy as np
import pandas as pd
from PIL import Image
import torchxrayvision as xrv

import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms
import torch.nn as nn

# -------------------------
# 1) Download dataset
# -------------------------
root = Path(kagglehub.dataset_download("nih-chest-xrays/data"))
print("Path to dataset files:", root)

Path to dataset files: /home/stachu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3


In [ ]:
KFOLDS = 5
THRESHOLD = 0.5
NUM_EPOCHS = 10
SUBSET_FRAC = None

In [3]:
candidate_csv = [
    root / "Data_Entry_2017.csv",
    root / "data" / "Data_Entry_2017.csv",
]
csv_path = next((p for p in candidate_csv if p.exists()), None)
if csv_path is None:
    csv_matches = list(root.rglob("Data_Entry_2017*.csv"))
    if not csv_matches:
        raise FileNotFoundError("Could not find Data_Entry_2017 CSV in downloaded dataset.")
    csv_path = csv_matches[0]

print("Using metadata:", csv_path)
df = pd.read_csv(csv_path)
df

Using metadata: /home/stachu/.cache/kagglehub/datasets/nih-chest-xrays/data/versions/3/Data_Entry_2017.csv


,Image Index,Finding Labels,Follow-up #,Patient ID,Patient Age,Patient Gender,View Position,OriginalImage[Width,Height],OriginalImagePixelSpacing[x,y],Unnamed: 11
0,00000001_000.png,Cardiomegaly,0,1,58,M,PA,2682,2749,0.143,0.143,NaN
1,00000001_001.png,Cardiomegaly|Emphysema,1,1,58,M,PA,2894,2729,0.143,0.143,NaN
2,00000001_002.png,Cardiomegaly|Effusion,2,1,58,M,PA,2500,2048,0.168,0.168,NaN
3,00000002_000.png,No Finding,0,2,81,M,PA,2500,2048,0.171,0.171,NaN
4,00000003_000.png,Hernia,0,3,81,F,PA,2582,2991,0.143,0.143,NaN
...,...,...,...,...,...,...,...,...,...,...,...,...
112115,00030801_001.png,Mass|Pneumonia,1,30801,39,M,PA,2048,2500,0.168,0.168,NaN
112116,00030802_000.png,No Finding,0,30802,29,M,PA,2048,2500,0.168,0.168,NaN
112117,00030803_000.png,No Finding,0,30803,42,F,PA,2048,2500,0.168,0.168,NaN
112118,00030804_000.png,No Finding,0,30804,30,F,PA,2048,2500,0.168,0.168,NaN


In [4]:
if "Image Index" not in df.columns:
    alt_cols = [c for c in df.columns if c.lower().replace("_", " ") == "image index"]
    if alt_cols:
        df = df.rename(columns={alt_cols[0]: "Image Index"})
if "Finding Labels" not in df.columns:
    alt_cols = [c for c in df.columns if c.lower().replace("_", " ") == "finding labels"]
    if alt_cols:
        df = df.rename(columns={alt_cols[0]: "Finding Labels"})

if "Image Index" not in df.columns or "Finding Labels" not in df.columns:
    raise ValueError("CSV must contain 'Image Index' and 'Finding Labels' columns.")

In [5]:
img_exts = {".png", ".jpg", ".jpeg"}
image_files = [p for p in root.rglob("*") if p.suffix.lower() in img_exts]
if not image_files:
    raise FileNotFoundError("No image files found. Make sure the dataset extracted correctly.")

name_to_path = {}
for p in image_files:
    name_to_path.setdefault(p.name, p)

df["image_path"] = df["Image Index"].map(lambda x: str(name_to_path.get(str(x), "")))
df = df[df["image_path"] != ""].reset_index(drop=True)

if len(df) == 0:
    raise RuntimeError("No metadata rows matched image files.")


print(f"Matched rows with images: {len(df):,}")

Matched rows with images: 112,120


In [6]:
from sklearn.preprocessing import MultiLabelBinarizer
import numpy as np

id_df = df[df["Finding Labels"] != "No Finding"].copy()
ood_df = df[df["Finding Labels"] == "No Finding"].copy()

# Extract all unique diseases split by '|'
id_df['finding_list'] = id_df['Finding Labels'].apply(lambda x: str(x).split('|'))

# Fit MultiLabelBinarizer to create binary arrays for each class
mlb = MultiLabelBinarizer()
targets = mlb.fit_transform(id_df['finding_list'])

# Add target column as float32 arrays suitable for PyTorch BCEWithLogitsLoss
id_df['target'] = list(targets.astype(np.float32))

NUM_CLASSES = len(mlb.classes_)
LABEL_NAMES = list(mlb.classes_)

# OOD samples (No Finding) have 0 for all disease classes
ood_df['target'] = [np.zeros(NUM_CLASSES, dtype=np.float32)] * len(ood_df)

print(f"Number of classes: {NUM_CLASSES}")
print(f"Classes: {LABEL_NAMES}")

Number of classes: 14
Classes: ['Atelectasis', 'Cardiomegaly', 'Consolidation', 'Edema', 'Effusion', 'Emphysema', 'Fibrosis', 'Hernia', 'Infiltration', 'Mass', 'Nodule', 'Pleural_Thickening', 'Pneumonia', 'Pneumothorax']


In [7]:
from sklearn.model_selection import train_test_split

if SUBSET_FRAC is not None:
    id_df = id_df.sample(frac=SUBSET_FRAC, random_state=42).reset_index(drop=True)

# Hold out exactly 10% of the dataset for probability calibration first.
# The remaining 90% will be used for our 5-Fold Cross Validation.
X_cv, X_test = train_test_split(
    id_df,
    test_size=0.1,
    random_state=42,
)

# Reset index for clean KFold splitting later
X_cv = X_cv.reset_index(drop=True)

print(f"Available for 5-Fold CV: {len(X_cv):,}")
print(f"Calibration images: {len(X_test):,}")

Available for 5-Fold CV: 46,583
Calibration images: 5,176


In [8]:

# -------------------------
# 7) PyTorch Dataset + DataLoader (class-specific)
# -------------------------
IMAGE_SIZE = 224
BATCH_SIZE = 64
NUM_WORKERS = 0

# Data Augmentation for training
train_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE + 32, IMAGE_SIZE + 32)),
    transforms.RandomCrop(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(15),
    transforms.ColorJitter(brightness=0.2, contrast=0.2),
    transforms.RandomAffine(degrees=0, translate=(0.05, 0.05), scale=(0.95, 1.05)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2048 - 1024)
])

val_tfms = transforms.Compose([
    transforms.Grayscale(num_output_channels=1),
    transforms.Resize((IMAGE_SIZE, IMAGE_SIZE)),
    transforms.ToTensor(),
    transforms.Lambda(lambda x: x * 2048 - 1024)
])


class NIHChestDataset(Dataset):
    def __init__(self, frame: pd.DataFrame, transform=None):
        self.frame = frame
        self.transform = transform

    def __len__(self):
        return len(self.frame)

    def __getitem__(self, idx):
        row = self.frame.iloc[idx]
        img = Image.open(row["image_path"]).convert("RGB")
        if self.transform is not None:
            img = self.transform(img)
        target = torch.tensor(row["target"], dtype=torch.float32)
        return img, target


# We use X_cv and X_cal here for sanity check DataLoaders. 
# Real training DataLoaders will be created inside the K-Fold loop.
train_ds = NIHChestDataset(X_cv, transform=train_tfms)
cal_ds = NIHChestDataset(X_test, transform=val_tfms)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

test_loader = DataLoader(
    cal_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,  # Don't shuffle validation
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print("Class-specific DataLoaders ready.")
print("Available loaders: train_loader, val_loader")

# Optional sanity check
x0, y0 = next(iter(train_loader))
x1, y1 = next(iter(test_loader))
print("Train batch:", tuple(x0.shape), tuple(y0.shape), "mean target:", float(y0.mean()), "mean:", float(x0.mean()))
print("Validation batch:", tuple(x1.shape), tuple(y1.shape), "mean target:", float(y1.mean()))


Class-specific DataLoaders ready.
Available loaders: train_loader, val_loader
Train batch: (64, 1, 224, 224) (64, 14) mean target: 0.1071428582072258 mean: -67.38931274414062
Validation batch: (64, 1, 224, 224) (64, 14) mean target: 0.1194196417927742


In [9]:
# from sklearn.metrics import roc_auc_score
# from tqdm import tqdm
# import numpy as np

# # torchxrayvision models output specific 18 pathologies. We need to align our 14 dataset labels with them.
# pathologies = model.pathologies

# class_mapping = []
# for label in LABEL_NAMES:
#     normalized_label = label.replace("_", " ").lower()
#     matched_idx = -1
#     for i, path in enumerate(pathologies):
#         if normalized_label == path.lower():
#             matched_idx = i
#             break
#     class_mapping.append(matched_idx)

# model.eval()
# all_preds = []
# all_targets = []

# with torch.no_grad():
#     for images, targets in tqdm(val_loader, desc="Validating Pretrained Model"):
#         # xrv DenseNet expects 1-channel images, but our dataloader returns 3-channel RGB.
#         # We average across the channels to convert to Grayscale.
#         images_1c = images.mean(dim=1, keepdim=True).to(device)
        
#         outputs = model(images_1c)
#         all_preds.append(outputs.cpu().numpy())
#         all_targets.append(targets.numpy())

# all_preds = np.concatenate(all_preds, axis=0)
# all_targets = np.concatenate(all_targets, axis=0)

# print("\n--- Pretrained Model AUROC on Validation Set ---")
# valid_aurocs = []

# for i, label in enumerate(LABEL_NAMES):
#     pretrain_idx = class_mapping[i]
#     if pretrain_idx != -1:
#         if len(np.unique(all_targets[:, i])) > 1:
#             auc = roc_auc_score(all_targets[:, i], all_preds[:, pretrain_idx])
#             valid_aurocs.append(auc)
#             print(f"{label:20s}: {auc:.4f}")
#     else:
#         print(f"{label:20s}: Not predicted by model")

# if valid_aurocs:
#     print(f"\nMean AUROC across matched classes: {np.mean(valid_aurocs):.4f}")

In [10]:
device = "cuda" if torch.cuda.is_available() else "cpu"

def build_model():
    model = xrv.models.DenseNet(weights='densenet121-res224-all', drop_rate=0.2).to(device)

    # Freeze all early layers for transfer learning
    for param in model.parameters():
        param.requires_grad = False

    # Optional: Unfreeze the last blocks for fine-tuning
    # For DenseNet, this is 'features.denseblock4'
    # for param in model.features.denseblock4.parameters():
    #     param.requires_grad = True

    # Replace the classification head (DenseNet uses 'classifier', not 'fc')
    in_features = model.classifier.in_features
    model.classifier = nn.Linear(in_features, NUM_CLASSES)
    model.op_threshs = None
    model = model.to(device)

    return model


In [11]:
# Cell 8: Train/validation loop utilities
from tqdm.auto import tqdm


def binary_metrics(logits: torch.Tensor, targets: torch.Tensor, threshold: float = 0.5, eps: float = 1e-8):
    probs = torch.sigmoid(logits)
    preds = (probs >= threshold).float()

    tp = (preds * targets).sum()
    fp = (preds * (1 - targets)).sum()
    fn = ((1 - preds) * targets).sum()
    tn = ((1 - preds) * (1 - targets)).sum()

    accuracy = (tp + tn) / (tp + tn + fp + fn + eps)
    precision = tp / (tp + fp + eps)
    recall = tp / (tp + fn + eps)
    f1 = (2 * precision * recall) / (precision + recall + eps)
    return accuracy.item(), f1.item()

def run_epoch(model, optimizer, criterion, loader, train_mode=True):
    model.train() if train_mode else model.eval()

    total_loss = 0.0
    total_acc = 0.0
    total_f1 = 0.0
    steps = 0

    pbar = tqdm(loader, desc="Train" if train_mode else "Val", leave=False)
    for images, targets in pbar:
        images = images.to(device, non_blocking=True)
        targets = targets.to(device, non_blocking=True)

        with torch.set_grad_enabled(train_mode):
            logits = model(images)
            loss = criterion(logits, targets)

            if train_mode:
                optimizer.zero_grad(set_to_none=True)
                loss.backward()
                optimizer.step()

        acc, f1 = binary_metrics(logits.detach(), targets, threshold=THRESHOLD)
        
        total_loss += loss.item()
        total_acc += acc
        total_f1 += f1
        steps += 1
        
        pbar.set_postfix(loss=f"{loss.item():.4f}", f1=f"{f1:.4f}")

    return (
        total_loss / max(steps, 1),
        total_acc / max(steps, 1),
        total_f1 / max(steps, 1),
    )

In [12]:
from sklearn.model_selection import KFold

# 5-Fold Cross Validation Setup
kf = KFold(n_splits=KFOLDS, shuffle=True, random_state=42)

fold_histories = []
best_fold_f1s = []

print(f"Starting {KFOLDS}-Fold Cross Validation...")

for fold, (train_idx, val_idx) in enumerate(kf.split(X_cv)):
    print(f"\n{'='*20} FOLD {fold + 1}/{KFOLDS} {'='*20}")
    
    # 1. Split Data
    fold_train_df = X_cv.iloc[train_idx]
    fold_val_df = X_cv.iloc[val_idx]
    
    # 2. Setup DataLoaders
    train_ds = NIHChestDataset(fold_train_df, transform=train_tfms)
    val_ds = NIHChestDataset(fold_val_df, transform=val_tfms)
    
    train_loader = DataLoader(train_ds, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    val_loader = DataLoader(val_ds, batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS, pin_memory=torch.cuda.is_available())
    
    # 3. Initialize Fresh Model
    model = build_model()
    
    # 4. Handle Imbalance for Current Fold
    train_targets = np.stack(fold_train_df["target"].values).astype(np.float32)
    pos_count = train_targets.sum(axis=0)
    neg_count = len(train_targets) - pos_count
    pos_weight_value = torch.tensor(np.divide(neg_count, np.maximum(pos_count, 1.0)), dtype=torch.float32, device=device)
    
    criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight_value)
    
    optimizer = torch.optim.AdamW(filter(lambda p: p.requires_grad, model.parameters()), lr=1e-4, weight_decay=1e-2)
    
    # 5. Train Model
    best_val_f1 = -1.0
    history = []
    
    for epoch in range(1, NUM_EPOCHS + 1):
        train_loss, train_acc, train_f1 = run_epoch(model, optimizer, criterion, train_loader, train_mode=True)
        val_loss, val_acc, val_f1 = run_epoch(model, optimizer, criterion, val_loader, train_mode=False)

        history.append({
            "epoch": epoch,
            "train_loss": train_loss,
            "train_acc": train_acc,
            "train_f1": train_f1,
            "val_loss": val_loss,
            "val_acc": val_acc,
            "val_f1": val_f1,
        })

        print(
            f"Fold {fold+1} Epoch [{epoch}/{NUM_EPOCHS}] "
            f"train_loss={train_loss:.4f} train_f1={train_f1:.4f} | "
            f"val_loss={val_loss:.4f} val_f1={val_f1:.4f}"
        )

        if val_f1 > best_val_f1:
            best_val_f1 = val_f1
            torch.save(model.state_dict(), f"best_densenet50_fold_{fold+1}.pth")
            print(f"  -> Saved best model for Fold {fold+1} (val_f1={best_val_f1:.4f})")
            
    fold_histories.append(history)
    best_fold_f1s.append(best_val_f1)
    print(f"Fold {fold+1} finished. Best F1: {best_val_f1:.4f}")

print("\n" + "="*30)
print(f"Cross Validation Complete! Mean Best F1: {np.mean(best_fold_f1s):.4f}")

Starting 2-Fold Cross Validation...

==================== FOLD 1/2 ====================


Train:   0%|          | 0/364 [00:00<?, ?it/s]

Val:   0%|          | 0/364 [00:00<?, ?it/s]

Fold 1 Epoch [1/5] train_loss=1.2139 train_f1=0.2380 | val_loss=1.1749 val_f1=0.3015
  -> Saved best model for Fold 1 (val_f1=0.3015)


Train:   0%|          | 0/364 [00:00<?, ?it/s]

...

Val:   0%|          | 0/364 [00:00<?, ?it/s]

Fold 2 Epoch [3/5] train_loss=1.1756 train_f1=0.2798 | val_loss=1.1407 val_f1=0.3057
  -> Saved best model for Fold 2 (val_f1=0.3057)


Train:   0%|          | 0/364 [00:00<?, ?it/s]

Val:   0%|          | 0/364 [00:00<?, ?it/s]

KeyboardInterrupt: 

In [14]:
import pickle

with open("densenet121-training-histories.pkl", "wb") as fp:
    pickle.dump(fold_histories, fp)

In [15]:
def calibrate(model):
    model.eval()

    for module in model.modules():
        if module.__class__.__name__ == '_DenseLayer':
            module.training = True

In [16]:
from tqdm import tqdm

def entropy(p):
    # Add epsilon to prevent log(0). 
    # Computes per-sample entropy across classes. Shape: (batch_size,)
    eps = 1e-8
    return -torch.sum(p * torch.log(p + eps), dim=-1)

def epistemic_uncertainty(preds):
    # preds shape: (N_models_or_passes, batch_size, num_classes)
    
    # 1. Total entropy: entropy of the mean prediction
    p_mean = torch.mean(preds, dim=0) # (batch_size, num_classes)
    tot_ent = entropy(p_mean)         # (batch_size,)
    
    # 2. Aleatoric uncertainty: mean of the entropies
    aleatoric = torch.mean(entropy(preds), dim=0) # (batch_size,)
    
    # 3. Epistemic uncertainty
    return tot_ent - aleatoric

def compute_id_scores(id_loader, num_passes=10):
    # Pre-load models ONCE outside the batch loop to prevent massive disk I/O bottlenecks!
    models = []
    for fold in range(KFOLDS):
        m = build_model()
        m.load_state_dict(torch.load(f"best_densenet50_fold_{fold+1}.pth", map_location=device))
        calibrate(m)
        models.append(m)

    id_scores = []
    with torch.no_grad():
        for images, _ in tqdm(id_loader, "Computing ID scores"):
            images = images.to(device)
            
            outputs = []
            for m in models:
                # MCDropout: Pass data multiple times through the SAME model
                for _ in range(num_passes):
                    logits = m(images)
                    probs = torch.sigmoid(logits) # Convert logits to probabilities!
                    outputs.append(probs)
            
            # Stack all multi-model outputs 
            # New shape: (KFOLDS * num_passes, batch_size, num_classes)
            preds = torch.stack(outputs)
            
            # Vectorized epistemic uncertainty -> shape: (batch_size,)
            uncertainty = epistemic_uncertainty(preds)
            
            # Negative uncertainty is used so higher score = more ID (less uncertain)
            id_scores.extend(-uncertainty.cpu().numpy())
            
    return id_scores

In [17]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

id_scores = compute_id_scores(test_loader)

Computing ID scores: 100%|██████████| 81/81 [10:53<00:00,  8.06s/it]


In [ ]:
id_scores

[np.float32(-0.0009851456),
 np.float32(-0.0044059753),
 np.float32(-0.0008883476),
 np.float32(-0.00106287),
 np.float32(-0.002471447),
 np.float32(-0.00071525574),
 np.float32(-0.0020303726),
 np.float32(-0.0016412735),
 np.float32(-0.0011520386),
 np.float32(-0.001557827),
 np.float32(-0.0009999275),
 np.float32(-0.0010514259),
 np.float32(-0.0025525093),
 np.float32(-0.0010757446),
 np.float32(-0.0014977455),
 np.float32(-0.0009737015),
 np.float32(-0.0017943382),
 np.float32(-0.00096178055),
 np.float32(-0.0008058548),
 np.float32(-0.0011725426),
 np.float32(-0.0008678436),
 np.float32(-0.004067898),
 np.float32(-0.0015563965),
 np.float32(-0.0053629875),
 np.float32(-0.0016317368),
 np.float32(-0.0010242462),
 np.float32(-0.0046420097),
 np.float32(-0.004123211),
 np.float32(-0.0008234978),
 np.float32(-0.0017232895),
 np.float32(-0.002175808),
 np.float32(-0.001522541),
 np.float32(-0.0013628006),
 np.float32(-0.0031166077),
 np.float32(-0.0008111),
 np.float32(-0.0021219254),
 

In [18]:
def determine_threshold(id_scores, percentile=5):
    threshold = np.percentile(id_scores, percentile)
    return threshold

threshold = determine_threshold(id_scores, percentile=5)
threshold

np.float32(-0.016758919)

In [19]:
df_ood_subset = ood_df.sample(n = len(X_test) // KFOLDS, random_state=42)
ood_dataset = NIHChestDataset(df_ood_subset, transform=val_tfms)
ood_loader = DataLoader(
    ood_dataset,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

In [20]:
ood_scores = compute_id_scores(ood_loader)

Computing ID scores: 100%|██████████| 41/41 [05:27<00:00,  8.00s/it]


In [21]:
from metrics import fpr
from sklearn.metrics import average_precision_score

y_true = [1] * len(id_scores) + [0] * len(ood_scores)
y_scores = id_scores + ood_scores

auprc = average_precision_score(y_true, y_scores)
fpr90 = fpr(y_true, y_scores, percentile=10)
fpr95 = fpr(y_true, y_scores, percentile=5)
fpr99 = fpr(y_true, y_scores, percentile=1)

print(f"AUPRC: {auprc:.4f}")
print(f"FPR at 90% TPR: {fpr90:.4f}")
print(f"FPR at 95% TPR: {fpr95:.4f}")
print(f"FPR at 99% TPR: {fpr99:.4f}")

AUPRC: 0.8075
FPR at 90% TPR: 0.7349
FPR at 95% TPR: 0.8640
FPR at 99% TPR: 0.9675


In [22]:
from torchvision import datasets, transforms
import torch
from torch.utils.data import DataLoader, Subset

# Ensure all Caltech images are forced to RGB and use identical validation transforms
caltech_tfms = transforms.Compose([
    transforms.Lambda(lambda x: x.convert("RGB")),
    val_tfms
])

caltech_data = datasets.Caltech256(
    root='./data', 
    download=True, 
    transform=caltech_tfms
)

# Randomly sample the Caltech dataset to match the size of your Validation (ID) dataset
num_samples = min(len(cal_ds), len(caltech_data))
indices = torch.randperm(len(caltech_data))[:num_samples]
caltech_subset = Subset(caltech_data, indices.tolist())

caltech_loader = DataLoader(
    caltech_subset,
    batch_size=16,
    shuffle=False,
    num_workers=NUM_WORKERS,
    pin_memory=torch.cuda.is_available(),
)

print(f"Loaded {len(caltech_subset)} Caltech256 images to use as OOD.")

Loaded 5176 Caltech256 images to use as OOD.


In [23]:
ood_scores_caltech = compute_id_scores(caltech_loader)

# Assign class labels exactly as in the previous cell
y_true_caltech = [1] * len(id_scores) + [0] * len(ood_scores_caltech)
y_scores_caltech = id_scores + ood_scores_caltech

auprc_caltech = average_precision_score(y_true_caltech, y_scores_caltech)
fpr90_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=10)
fpr95_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=5)
fpr99_caltech = fpr(y_true_caltech, y_scores_caltech, percentile=1)

print("=== ID (NIH X-Ray) vs OOD (Caltech256) ===")
print(f"AUPRC: {auprc_caltech:.4f}")
print(f"FPR at 90% TPR: {fpr90_caltech:.4f}")
print(f"FPR at 95% TPR: {fpr95_caltech:.4f}")
print(f"FPR at 99% TPR: {fpr99_caltech:.4f}")

Computing ID scores: 100%|██████████| 324/324 [12:10<00:00,  2.25s/it]

=== ID (NIH X-Ray) vs OOD (Caltech256) ===
AUPRC: 0.5147
FPR at 90% TPR: 0.8271
FPR at 95% TPR: 0.8688
FPR at 99% TPR: 0.9075


In [24]:
ood_scores_caltech

[np.float32(-0.015890598),
 np.float32(-0.0054512024),
 np.float32(-0.007381439),
 np.float32(-0.013905048),
 np.float32(-0.0062761307),
 np.float32(-0.0075101852),
 np.float32(-0.009744644),
 np.float32(-0.009439945),
 np.float32(-0.0028104782),
 np.float32(-0.014815807),
 np.float32(-0.0031218529),
 np.float32(-0.014636517),
 np.float32(-0.010910511),
 np.float32(-0.008180141),
 np.float32(-0.0040955544),
 np.float32(-0.016447544),
 np.float32(-0.0025367737),
 np.float32(-0.007449627),
 np.float32(-0.0057964325),
 np.float32(-0.0030078888),
 np.float32(-0.0046567917),
 np.float32(-0.0044288635),
 np.float32(-0.008577347),
 np.float32(-0.004087925),
 np.float32(-0.021622181),
 np.float32(-0.008120537),
 np.float32(-0.0028896332),
 np.float32(-0.006640911),
 np.float32(-0.008772373),
 np.float32(-0.009563446),
 np.float32(-0.0124435425),
 np.float32(-0.0024704933),
 np.float32(-0.008913517),
 np.float32(-0.010192394),
 np.float32(-0.008187771),
 np.float32(-0.004529476),
 np.float32(-0